## Experiment

In [14]:
## Experiment
spec_no = 1
config_file = f"config_spec_{spec_no}.yaml"

In [ ]:
import os
import json, shutil, subprocess, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from src import load_data, build_spec, compute_stats

# data and model from the PAC config (nothing else of it is used here)
GlobalHydra.instance().clear()
initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base="1.3")
cfg = compose(config_name=config_file)
NB_DIR = PROJECT_ROOT / "notebooks"
cfg.data.edges_path = str((NB_DIR / cfg.data.edges_path).resolve())
cfg.data.nodes_path = str((NB_DIR / cfg.data.nodes_path).resolve())
NODES = cfg.data.nodes_path
EDGES = cfg.data.edges_path

if spec_no != 1:
    cfg.data.edgecov = {k: str((NB_DIR / v).resolve()) for k, v in dict(cfg.data.edgecov).items()}
    COV_DIR = str(Path(cfg.data.edgecov["use"]).parent)
adj, nodes, ecov = load_data(cfg.data)
spec = build_spec(nodes, ecov, cfg.model, directed=True)
g_obs = compute_stats(adj, spec)
labels = list(spec.labels)

print("model:", cfg.model.statistic_terms)
print("g_obs =", dict(zip(labels, np.round(g_obs, 2))))

# ---- ERGM settings ----
NSIM, MAXD = cfg.ergm.nsim, cfg.ergm.maxd
MCMLE_ESS, MCMLE_MAXIT = cfg.ergm.mcmle_ess, cfg.ergm.mcmle_maxit
SIM_BURNIN, SIM_INTERVAL = cfg.ergm.sim_burnin, cfg.ergm.sim_interval 
SEED = cfg.ergm.seed

RSCRIPT = shutil.which("Rscript") or r"C:/Program Files/R/R-4.5.3/bin/Rscript.exe"
run_dir = PROJECT_ROOT / "output" / f"run_ERGM_spec_{spec_no}"
R_PATH = run_dir / "ergm_run.R"

os.makedirs(run_dir, exist_ok=True)

print("results ->", run_dir)

Network loaded — nodes: 79, edges: 781, edgecov: []
model: edges + nodematch(GroupC) + nodefactor(GroupC, base=3) + gwesp(0.25)
g_obs = {'edges': np.float64(781.0), 'nodematch.GroupC': np.float64(663.0), 'nodefactor.GroupC.1': np.float64(733.0), 'nodefactor.GroupC.2': np.float64(584.0), 'gwesp': np.float64(929.04)}
results -> /Users/giulia/Documents/Projects/Research/ExtraCanNullModels/enrico_repo/PAC_ERGM/output/run_ERGM_spec_1/ergm


## Fit and Simulate

- The R script is written into the run folder and run with `Rscript`.
- The output goes to `R_log.txt`. `ergm` reports a non-converged
MCMLE with a message (not a warning): the script catches it and writes it in the status. 
- If the MLE fails altogether, the script falls back to the MPLE, only to show what that model simulates.

In [16]:
if spec_no == 2:
    model_spec = "edges + mutual + gwesp(decay = 0.5, fixed = TRUE) + \
              edgecov(m_use) + edgecov(m_team) + edgecov(m_flr) + edgecov(m_met)" 
elif spec_no == 3:
    model_spec = "edges + mutual + gwesp(decay = 0.5, fixed = TRUE) + \
        gwodegree(decay = 0.5, fixed = TRUE) + gwidegree(decay = 0.5, fixed = TRUE) + \
        edgecov(m_use) + edgecov(m_team) + edgecov(m_flr) + edgecov(m_met)"

In [ ]:
if spec_no == 1:
    R_CODE = r'''
    suppressMessages({library(network); library(ergm)})
    a <- commandArgs(trailingOnly = TRUE)
    edges_csv <- a[1]; nodes_csv <- a[2];
    nsim <- as.integer(a[3]); maxesp <- as.integer(a[4])
    ess <- as.numeric(a[5]); maxit <- as.integer(a[6])
    burnin <- as.numeric(a[7]); interval <- as.numeric(a[8]); seed <- as.integer(a[9]);
    run_dir <- a[10]

    nodes <- read.csv(nodes_csv); ed <- read.csv(edges_csv)
    n <- nrow(nodes)
    net <- network.initialize(n, directed = TRUE)
    net[as.matrix(ed[, c("from","to")]) + 1] <- 1
    net %v% "GroupC" <- as.character(nodes$GroupC)

    form <- net ~ edges + nodematch("GroupC") +
            nodefactor("GroupC", base = 3) + gwesp(decay = 0.25, fixed = TRUE)

    gobs <- summary(form)
    write.csv(data.frame(term = names(gobs), value = as.numeric(gobs)),
            file.path(run_dir, "gobs.csv"), row.names = FALSE)

    # (1) MLE by MCMLE
    fit_status <- "CONVERGED"
    not_conv <- function(cond) if (grepl("did not converge", conditionMessage(cond)))
    fit_status <<- sub("^CONVERGED", "NOT CONVERGED", fit_status)
    t0 <- proc.time()[["elapsed"]]
    fit <- withCallingHandlers(
    tryCatch(ergm(form, control = control.ergm(seed = seed, MCMLE.effectiveSize = ess,
                                                MCMLE.maxit = maxit)),
            error = function(e){ fit_status <<- paste("DEGENERATE/ERROR:", conditionMessage(e)); NULL }),
    message = not_conv, warning = not_conv)
    fit_seconds <- proc.time()[["elapsed"]] - t0
    if (!is.null(fit)) fit_status <- paste0(fit_status, " (MCMLE iterations: ", fit$iterations, ")")
    writeLines(fit_status, file.path(run_dir, "status.txt"))
    writeLines(sprintf("%.1f", fit_seconds), file.path(run_dir, "fit_seconds.txt"))

    if (!is.null(fit)) {
    s  <- summary(fit)$coefficients
    co <- data.frame(term = rownames(s), estimate = s[,1], std_error = s[,2], p_value = s[, ncol(s)])
    writeLines(capture.output(print(summary(fit))), file.path(run_dir, "summary.txt"))
    fsim <- fit
    } else {
    fmple <- ergm(form, estimate = "MPLE")          # always exists: used to show the degeneracy
    co <- data.frame(term = names(coef(fmple)), estimate = as.numeric(coef(fmple)), std_error = NA, p_value = NA)
    fsim <- fmple
    }
    write.csv(co, file.path(run_dir, "coef.csv"), row.names = FALSE)

    # (2) simulate from the estimated model -> GoF on the sufficient statistics
    ctrl <- control.simulate.ergm(MCMC.burnin = burnin, MCMC.interval = interval)
    simstats <- simulate(fsim, nsim = nsim, output = "stats",   control = ctrl, seed = seed + 1)
    write.csv(as.data.frame(simstats), file.path(run_dir, "simstats.csv"), row.names = FALSE)
    # (3) GoF on ESP (OTP, as gwesp), which is NOT in the model
    simnets <- simulate(fsim, nsim = nsim, output = "network", control = ctrl, seed = seed + 1)
    m1 <- maxesp + 1
    espm <- matrix(0, nsim, m1)
    for (i in 1:nsim) {
    A <- as.matrix(simnets[[i]])
    S <- A %*% A; ev <- S[A == 1]
    espm[i,] <- if (length(ev)) tabulate(pmin(ev, maxesp) + 1, m1) else rep(0, m1)
    }
    write.csv(setNames(as.data.frame(espm), as.character(0:maxesp)),
            file.path(run_dir, "esp.csv"), row.names = FALSE)
    cat("DONE ukFaculty | MLE:", fit_status, "\n")
    '''
    cmd = [RSCRIPT, "--vanilla", str(R_PATH), EDGES, NODES, str(NSIM), str(MAXD),
           str(MCMLE_ESS), str(MCMLE_MAXIT), str(SIM_BURNIN), str(SIM_INTERVAL), 
           str(SEED), str(run_dir)]
else:
    R_CODE = r'''
    suppressMessages({library(network); library(ergm)})
    a <- commandArgs(trailingOnly = TRUE)
    edges_csv <- a[1]; cov_dir <- a[2]; model <- a[3];
    nsim <- as.integer(a[4]); maxd <- as.integer(a[5]); n <- as.integer(a[6])
    ess <- as.numeric(a[7]); maxit <- as.integer(a[8])
    burnin <- as.numeric(a[9]); interval <- as.numeric(a[10]); seed <- as.integer(a[11]);
    run_dir <- a[12]

    ed  <- read.csv(edges_csv)
    net <- network.initialize(n, directed = TRUE)
    net[as.matrix(ed[, c("from","to")]) + 1] <- 1

    rdm <- function(f) as.matrix(read.csv(file.path(cov_dir, f), header = FALSE))
    m_use <- rdm("use.csv"); m_team <- rdm("team.csv")
    m_flr <- rdm("floor.csv"); m_met <- rdm("metric.csv")

    form <- as.formula(
    paste("net ~", model),
    env = environment()
    )

    gobs <- summary(form)
    write.csv(data.frame(term = names(gobs), value = as.numeric(gobs)),
            file.path(run_dir, "gobs.csv"), row.names = FALSE)

    # ---- MLE by MCMLE (with degeneracy guard: if it fails we still get the MPLE) ----
    status <- "CONVERGED"
    not_conv <- function(cond) if (grepl("did not converge", conditionMessage(cond)))
    status <<- sub("^CONVERGED", "NOT CONVERGED", status)
    t0 <- proc.time()[["elapsed"]]
    fit <- withCallingHandlers(
    tryCatch(ergm(form, control = control.ergm(seed = seed, MCMLE.effectiveSize = ess, MCMLE.maxit = maxit)),
            error = function(e) { status <<- paste("ERROR:", conditionMessage(e)); NULL }),
    message = not_conv, warning = not_conv)
    writeLines(sprintf("%.1f", proc.time()[["elapsed"]] - t0), file.path(run_dir, "fit_seconds.txt"))
    if (is.null(fit)) {
    fsim <- ergm(form, estimate = "MPLE")
    co <- data.frame(term = names(coef(fsim)), estimate = as.numeric(coef(fsim)),
                    std_error = NA, p_value = NA)
    } else {
    status <- paste0(status, " (MCMLE iterations: ", fit$iterations, ")")
    fsim <- fit; s <- summary(fit)$coefficients
    co <- data.frame(term = rownames(s), estimate = s[,1], std_error = s[,2],
                    p_value = s[, ncol(s)])
    writeLines(capture.output(print(summary(fit))), file.path(run_dir, "summary.txt"))
    }
    write.csv(co, file.path(run_dir, "coef.csv"), row.names = FALSE)
    writeLines(status, file.path(run_dir, "status.txt"))

    # ---- simulate from the estimated model: the same draws as the PAC GoF ----
    ctrl <- control.simulate.ergm(MCMC.burnin = burnin, MCMC.interval = interval)
    simstats <- simulate(fsim, nsim = nsim, output = "stats",   control = ctrl, seed = seed + 1)
    simnets  <- simulate(fsim, nsim = nsim, output = "network", control = ctrl, seed = seed + 1)
    write.csv(as.data.frame(simstats), file.path(run_dir, "simstats.csv"), row.names = FALSE)

    m1 <- maxd + 1
    indeg <- matrix(0,nsim,m1); outdeg <- matrix(0,nsim,m1); espm <- matrix(0,nsim,m1)
    for (i in 1:nsim) {
    A <- as.matrix(simnets[[i]]); id <- colSums(A); od <- rowSums(A)
    indeg[i,]  <- tabulate(pmin(id,maxd)+1, m1)
    outdeg[i,] <- tabulate(pmin(od,maxd)+1, m1)
    S <- A %*% A; ev <- S[A == 1]
    espm[i,] <- if (length(ev)) tabulate(pmin(ev,maxd)+1, m1) else rep(0,m1)
    }
    cn <- as.character(0:maxd)
    write.csv(setNames(as.data.frame(indeg), cn),  file.path(run_dir, "indeg.csv"),  row.names=FALSE)
    write.csv(setNames(as.data.frame(outdeg),cn),  file.path(run_dir, "outdeg.csv"), row.names=FALSE)
    write.csv(setNames(as.data.frame(espm),  cn),  file.path(run_dir, "esp.csv"),    row.names=FALSE)

    cat("DONE |", status, "| observed edges:", gobs["edges"],
        "| median simulated edges:", median(simstats[,"edges"]), "\n")
    '''
    cmd = [RSCRIPT, "--vanilla", str(R_PATH), EDGES, COV_DIR, model_spec, 
           str(NSIM), str(MAXD),
           str(adj.shape[0]), str(MCMLE_ESS), 
           str(MCMLE_MAXIT), str(SIM_BURNIN),
           str(SIM_INTERVAL), str(SEED),
           str(run_dir)]
    
R_PATH.write_text(R_CODE, encoding="utf-8")
t0 = time.time()
with open(run_dir / "R_log.txt", "w", encoding="utf-8") as log:
    r = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, text=True)
run_seconds = time.time() - t0
print((run_dir / "R_log.txt").read_text(encoding="utf-8")[-1500:])
if r.returncode != 0:
    raise RuntimeError(f"Rscript failed (exit {r.returncode}): see {run_dir / 'R_log.txt'}")

status     = (run_dir / "off_status.txt").read_text().strip()
coef_ergm  = pd.read_csv(run_dir / "coef.csv")
gobs_R     = pd.read_csv(run_dir / "gobs.csv")["value"].values

assert np.allclose(gobs_R, g_obs, rtol=1e-4, atol=1e-3), f"R and Python g_obs differ: {gobs_R} vs {g_obs}"
print(f"ERGM status: {status} | total time {run_seconds / 60:.1f} min")
print(coef_ergm.to_string(index=False))

## Save Run Information (`run_info.json`)

In [4]:
info = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "model": str(cfg.model.statistic_terms),
    "status": status, "total_minutes": round(run_seconds / 60, 2),
    "fit_seconds": float((run_dir / "off_fit_seconds.txt").read_text()),
    "settings": {"nsim": NSIM, "maxd": MAXD, "MCMLE.effectiveSize": MCMLE_ESS,
                 "MCMLE.maxit": MCMLE_MAXIT, "MCMC.burnin (simulation)": SIM_BURNIN,
                 "MCMC.interval (simulation)": SIM_INTERVAL, "seed": SEED},
    "g_obs": dict(zip(labels, map(float, g_obs))),
    "coefficients": coef_ergm.to_dict(orient="records"),
}
with open(run_dir / "run_info.json", "w", encoding="utf-8") as f:
    json.dump(info, f, indent=2)
print("ERGM run saved in ->", run_dir)
for p in sorted(run_dir.rglob("*")):
    if p.is_file():
        print("   ", p.relative_to(run_dir))

ERGM run saved in -> C:\Users\UTENTE\Documents\pace\output\run_ERGM_office_uni2005
    ergm_office_uni2005.R
    figures\07_ergm_gof_stats.png
    figures\08_ergm_gof_degree_esp.png
    off_coef.csv
    off_esp.csv
    off_fit_seconds.txt
    off_gobs.csv
    off_indeg.csv
    off_outdeg.csv
    off_simstats.csv
    off_status.txt
    off_summary.txt
    R_log.txt
    run_info.json
